# 1 — Basic Local RAG Agent

The foundational LangGraph pattern: an LLM that decides whether to call a tool, calls it, then uses the retrieved context to answer — looping until it's done.

**What you'll learn**
- `MessagesState` — the built-in state type that accumulates a `messages` list for you
- `bind_tools(tools)` — tells the LLM what tools exist and what they do (via docstrings)
- `ToolNode` — handles tool dispatch automatically (no manual `if tool_name == ...`)
- `should_continue` with conditional edges — the core ReAct decision loop
- `MemorySaver` + `thread_id` — how multi-turn memory persists between `.invoke()` calls
- Why the graph cycles (agent → tools → agent) instead of being linear

**Companion examples:** 4-basic-react-agent (same loop pattern, more tools), 10-streaming-rag (stream updates instead of blocking invoke)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma chromadb python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Build a retrieval tool over inline documents ───────────────────────────
# The original script fetches live Python docs from the web on every run.
# Here we use inline documents so the notebook is fully self-contained.
#
# The @tool decorator does two things:
#   1. The DOCSTRING becomes the tool description the LLM reads
#   2. The SIGNATURE becomes the JSON schema the LLM uses to call it
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings

SAMPLE_DOCS = [
    Document(
        page_content="A Python list is an ordered, mutable collection of items. Create one with square brackets: my_list = [1, 2, 3]. Lists support indexing (my_list[0] returns 1) and negative indexing (my_list[-1] returns 3)."
    ),
    Document(
        page_content="List methods: append(item) adds to end, extend(iterable) adds multiple items, insert(i, item) adds at position, remove(item) removes first match, pop() removes and returns last item."
    ),
    Document(
        page_content="List slicing: my_list[start:stop:step]. Examples: my_list[1:3] returns elements at index 1 and 2. my_list[::-1] reverses the list. my_list[::2] returns every other element."
    ),
    Document(
        page_content="List comprehensions create new lists in one line: squares = [x**2 for x in range(10)]. Add a filter: evens = [x for x in range(20) if x % 2 == 0]."
    ),
    Document(
        page_content="Common list functions: len(lst) returns length, sorted(lst) returns a new sorted list, min(lst) and max(lst) return extremes, sum(lst) totals numeric items."
    ),
]

_store = Chroma.from_documents(SAMPLE_DOCS, OpenAIEmbeddings())
_retriever = _store.as_retriever(search_kwargs={"k": 2})


@tool
def retrieve_context(query: str) -> str:
    """Search the Python knowledge base for relevant documentation."""
    docs = _retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs) or "No results found."


print("Tool test:", retrieve_context.invoke({"query": "how to slice a list"})[:80], "...")

In [ ]:
# ── 4. Build the ReAct graph ───────────────────────────────────────────────────
# ReAct = Reason + Act. The pattern:
#   agent sees messages, either calls a tool or answers
#   tools executes whichever tool the agent called
#   back to agent with the tool result added to messages
#   repeats until agent decides to answer (no more tool calls)
from typing import Literal

from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

tools = [retrieve_context]
tool_node = ToolNode(tools)  # handles tool dispatch automatically

# bind_tools makes the LLM aware of the tool schema
model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)


def should_continue(state: MessagesState) -> Literal["tools", END]:
    """Route: if the last message has tool_calls, go to tools. Otherwise finish."""
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END


def call_model(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", call_model)
graph.add_node("tools", tool_node)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)  # decision point
graph.add_edge("tools", "agent")  # cycle back after tool execution

# MemorySaver + thread_id = multi-turn conversation memory
app = graph.compile(checkpointer=MemorySaver())

print("Graph: START -> agent <-> tools -> END")

In [ ]:
# ── 5. Ask a question — watch the tool call happen ────────────────────────────
# thread_id ties this and future .invoke() calls to the same memory slot.
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": 42}}

result = app.invoke(
    {"messages": [HumanMessage(content="Explain what a list is in Python")]},
    config=config,
)

# Print full message trace to see the tool call loop
for msg in result["messages"]:
    role = type(msg).__name__
    content = str(msg.content)[:150] if msg.content else ""
    print(f"[{role}] {content}")

In [ ]:
# ── 6. Multi-turn: ask a follow-up using the same thread_id ───────────────────
# MemorySaver persists the conversation — the agent remembers the first answer.
result2 = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="Give me a list comprehension example from what you just explained."
            )
        ]
    },
    config=config,  # same thread_id = same memory
)
print(result2["messages"][-1].content)

## Exercises

**Exercise 1 — Change the topic:** Replace `SAMPLE_DOCS` with documents about Python dictionaries. Update the `retrieve_context` docstring to match. Ask "What is a dictionary and how do I iterate over it?"

**Exercise 2 — Skip the tool:** Ask a question the LLM can answer without retrieval, like `"What is 2 + 2?"`. Does the graph still call the tool? Why or why not?

**Exercise 3 — Add a second tool:** Create a `@tool` function `calculator(expression: str)` that evaluates math. Add it to `tools` and re-compile the graph. Ask `"How many items are in a list of size 5 after you append 3 more?"` — does the agent call the right tool?

**Exercise 4 — Different thread:** Change `thread_id` to a new value (e.g. `99`) and re-run the follow-up question. Does it still work, or does the agent lose context? Why?

## What's next

- **4-basic-react-agent** — same ReAct loop with multiple tools
- **10-streaming-rag** — stream graph updates node-by-node instead of blocking until END
- **13-structured-output** — LLM extracts typed fields from retrieved docs